# Dwarf Galaxy $\langle\sigma v\rangle$ Upper-Limit Calculator

Derives 95% C.L. upper limits on the DM annihilation cross-section  
$\langle\sigma v\rangle$ by applying a Poisson log-likelihood ratio test.

The likelihood for a single energy bin with $N_{\rm off}$ observed (background)  
events and an expected signal $S = \langle\sigma v\rangle \cdot J \cdot {\rm gen\_ns}$:

$$
\mathcal{L} = \frac{(B+S)^N\, e^{-(B+S)}}{N!}
$$

The test statistic (Wilks):
$$
TS = -2\ln\!\left(\frac{\mathcal{L}}{\mathcal{L}_0}\right)
$$

where $\mathcal{L}_0$ is the null-signal likelihood.  
The 95% C.L. upper limit corresponds to $TS = 2.71$.

**`gen_ns`** is loaded from the `results/gen_ns/` folder produced by  
`SWGO_Gen_Ns.ipynb`. It encodes the expected signal **per unit**  
$\langle\sigma v\rangle \times J$, so that:

$$
N_{\rm signal} = \langle\sigma v\rangle \cdot J_{\rm source} \cdot {\rm gen\_ns}
$$

**Dwarf data** (J-factors, integration angles, etc.) come from  
Geringer-Sameth et al. 2015 (ApJ 801, 74): <https://iopscience.iop.org/article/10.1088/0004-637X/801/2/74>


## Imports

In [1]:
import numpy as np
from scipy import integrate
from scipy.interpolate import interp1d
from joblib import Parallel, delayed
import os


## ⚙️ Input Cell — configure everything here


In [2]:
# ── Analysis settings ─────────────────────────────────────────────────────────
CHANNEL      = 'b'    # must match a file in GENNS_DIR
INITIAL_MASS = 500    # [GeV] lowest DM mass to include (SWGO energy threshold)

# ── Paths ─────────────────────────────────────────────────────────────────────
MASS_FILE   = 'mass.txt'
BKG_FILE    = 'bkg.txt'
GENNS_DIR   = 'results/gen_ns'

# Dwarf data (Geringer-Sameth et al. 2015)
ANGLES_FILE = 'dwarfs/anglesSWGO.txt'      # integration angles [deg]
ORDER_FILE  = 'dwarfs/table1SWGO.txt'       # dwarf names (defines ordering)
J_TABLE     = 'dwarfs/table3complete.txt'   # J-factor table (full)

# Output
RESULTS_DIR = 'results/sigma'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Parallelisation ───────────────────────────────────────────────────────────
N_JOBS = -1   # -1 → all available cores


## Load data

In [3]:
# ── DM masses [GeV] ───────────────────────────────────────────────────────────
mass = np.loadtxt(MASS_FILE)
initial_mass_idx = list(mass).index(INITIAL_MASS)
mass_used = mass[initial_mass_idx:]   # only masses above SWGO threshold
print(f"DM mass range: {mass_used[0]:.0f} – {mass_used[-1]:.0f} GeV  ({len(mass_used)} points)")

# ── Background counts per steradian per energy bin ────────────────────────────
# Columns: E_min [GeV], E_max [GeV], background counts/sr (time-integrated)
bkg = np.loadtxt(BKG_FILE)
print(f"Background bins: {len(bkg)}")

# ── gen_ns  [dimensionless / (cm^3/s × GeV^2/cm^5)] ─────────────────────────
# Shape: (n_bkg_bins, n_mass_total)
gen_ns_all = np.loadtxt(os.path.join(GENNS_DIR, f'{CHANNEL}.txt'))
gen_ns = gen_ns_all[:, initial_mass_idx:]   # trim to INITIAL_MASS
print(f"gen_ns shape (trimmed): {gen_ns.shape}")

# ── Dwarf integration angles & J-factors (Geringer-Sameth et al. 2015) ────────
# table1SWGO.txt — ordering of dwarfs (name + something else)
order = np.loadtxt(ORDER_FILE, usecols=(0,), dtype=object)
n_dwarfs = len(order)
print(f"\nNumber of dwarf galaxies: {n_dwarfs}")
print("Dwarfs:", list(order))

# table3complete.txt — full J-factor table (skiprows=42 to skip header)
# Columns (0-based): name, angle, ..., logJ_low, logJ_med, logJ_high, ...
j_info = np.loadtxt(J_TABLE, skiprows=42, usecols=(0,1,2,3,4,5,6), dtype=object)

# Extract central log10(J) and asymmetric uncertainties for each dwarf,
# using the 0.5-deg integration angle row (index 27 in each dwarf's block).
logj  = np.array([float(j_info[order[i] == j_info[:,0]][27, 4]) for i in range(n_dwarfs)])
sigma_j = np.array([
    np.max([
        abs(float(j_info[order[i] == j_info[:,0]][27, 4]) - float(j_info[order[i] == j_info[:,0]][27, 3])),
        abs(float(j_info[order[i] == j_info[:,0]][27, 4]) - float(j_info[order[i] == j_info[:,0]][27, 5]))
    ])
    for i in range(n_dwarfs)
])

j_factor = 10 ** logj   # [GeV^2 / cm^5]
print(f"\nJ-factors [GeV^2/cm^5]:")
for name, lj in zip(order, logj):
    print(f"  {name:20s}  log10(J) = {lj:.2f}")

# ── Solid angle of the region of interest ─────────────────────────────────────
# We use a common integration angle taken from the first dwarf (all share 0.5 deg).
def solid_angle(theta_max_deg):
    """
    Compute the solid angle [sr] of a cone with half-opening angle theta_max.
    Ω = 2π ∫₀^θ sin(θ) dθ
    """
    theta_max_rad = theta_max_deg * np.pi / 180.0
    return 2 * np.pi * integrate.quad(np.sin, 0, theta_max_rad)[0]

angle_deg   = float(j_info[order[0] == j_info[:,0]][27, 1])  # integration angle [deg]
omega_roi   = solid_angle(angle_deg)   # [sr]
print(f"\nIntegration angle: {angle_deg:.2f} deg  →  solid angle: {omega_roi:.4e} sr")


DM mass range: 500 – 100000 GeV  (30 points)
Background bins: 35
gen_ns shape (trimmed): (35, 30)

Number of dwarf galaxies: 14
Dwarfs: ['Carina', 'Fornax', 'LeoI', 'LeoII', 'Sculptor', 'Sextans', 'BootesI', 'ComaBerenices', 'Hercules', 'LeoIV', 'LeoV', 'LeoT', 'Segue1', 'Segue2']

J-factors [GeV^2/cm^5]:
  Carina                log10(J) = 17.70
  Fornax                log10(J) = 17.71
  LeoI                  log10(J) = 17.70
  LeoII                 log10(J) = 17.97
  Sculptor              log10(J) = 18.37
  Sextans               log10(J) = 17.08
  BootesI               log10(J) = 17.96
  ComaBerenices         log10(J) = 18.90
  Hercules              log10(J) = 16.83
  LeoIV                 log10(J) = 16.32
  LeoV                  log10(J) = 16.37
  LeoT                  log10(J) = 17.11
  Segue1                log10(J) = 19.24
  Segue2                log10(J) = 16.21

Integration angle: 0.21 deg  →  solid angle: 4.3884e-05 sr


## Test-statistic functions

### Single-dwarf TS


In [4]:
def sing_TS(sigmav, Ns, Noff, k):
    """
    Poisson log-likelihood ratio test statistic for a single dwarf.

    TS = -2 Σ_bins [ N_off · ln(N_off + sigmav · Ns_k) - N_off · ln(N_off) - sigmav · Ns_k ]

    Parameters
    ----------
    sigmav : float  [cm^3/s]  DM annihilation cross-section (trial value)
    Ns     : ndarray, shape (n_bins,)  expected signal counts (= J × gen_ns[:,k])
    Noff   : ndarray, shape (n_bins,)  observed background counts
    k      : int    mass index (0 = INITIAL_MASS)

    Returns
    -------
    float  TS value (≥ 0; clipped to 0 for numerical noise near the minimum)
    """
    # Signal term per energy bin
    s = Noff * np.log(Noff + sigmav * Ns[:, k]) - Noff * np.log(Noff) - sigmav * Ns[:, k]
    TS = -2.0 * np.sum(s)
    return TS if TS > 5e-5 else 0.0


### Combined TS (all dwarfs simultaneously)

In [5]:
def combined_TS(sigmav, Ns, Noff, k):
    """
    Combined Poisson TS summed over all dwarfs.

    Parameters
    ----------
    sigmav : float  [cm^3/s]
    Ns     : ndarray, shape (n_bins, n_dwarfs, n_mass)
    Noff   : ndarray, shape (n_bins, n_dwarfs)
    k      : int  mass index

    Returns
    -------
    float  combined TS value
    """
    s = (Noff * np.log(Noff + sigmav * Ns[:, :, k])
         - Noff * np.log(Noff)
         - sigmav * Ns[:, :, k])
    TS = -2.0 * np.sum(s)
    return TS if TS > 5e-5 else 0.0


### Limit-finding helpers

For each DM mass, scan over $\langle\sigma v\rangle$ values and interpolate  
to find where $TS = 2.71$ (95% C.L. upper limit).


In [6]:
def _single_dwarf_limit_for_mass(k, Ns_j, Noff_j, sigmas):
    """
    Find the single-dwarf 95% CL upper limit for mass index k.
    Called in parallel inside sing_sigma_calc.
    """
    TS_values = np.array([sing_TS(sv, Ns_j, Noff_j, k) for sv in sigmas])
    # Only use the monotone portion to avoid interpolation failures
    f = interp1d(TS_values, sigmas)
    return float(f(2.71))


def _combined_limit_for_mass(k, Ns, Noff, sigmas):
    """
    Find the combined 95% CL upper limit for mass index k.
    Called in parallel inside sigma_calc.
    """
    TS_values = np.array([combined_TS(sv, Ns, Noff, k) for sv in sigmas])
    f = interp1d(TS_values, sigmas)
    return float(f(2.71))


def sing_sigma_calc(Ns_j, Noff_j):
    """
    Compute sigma_v 95% CL upper limits for a single dwarf over all masses.

    Parameters
    ----------
    Ns_j   : ndarray, shape (n_bins, n_mass)  expected signal
    Noff_j : ndarray, shape (n_bins,)         background counts

    Returns
    -------
    sigma : ndarray, shape (n_mass,)  [cm^3/s]
    """
    sigmas = np.geomspace(1e-25, 1e-10, 10_000)   # trial cross-sections [cm^3/s]
    limits = Parallel(n_jobs=N_JOBS)(
        delayed(_single_dwarf_limit_for_mass)(k, Ns_j, Noff_j, sigmas)
        for k in range(len(mass_used))
    )
    return np.array(limits)


def sigma_calc(Ns, Noff):
    """
    Compute combined sigma_v 95% CL upper limits over all masses.

    Parameters
    ----------
    Ns   : ndarray, shape (n_bins, n_dwarfs, n_mass)
    Noff : ndarray, shape (n_bins, n_dwarfs)

    Returns
    -------
    sigma : ndarray, shape (n_mass,)  [cm^3/s]
    """
    sigmas = np.geomspace(1e-28, 1e-22, 10_000)   # tighter range for combined
    limits = Parallel(n_jobs=N_JOBS)(
        delayed(_combined_limit_for_mass)(k, Ns, Noff, sigmas)
        for k in range(len(mass_used))
    )
    return np.array(limits)


## Build signal and background arrays

For each dwarf and each energy bin:
- `Ns[i, j, k]` = expected signal counts  
  = `j_factor[j] × gen_ns[i, k]`  (still needs to be multiplied by $\langle\sigma v\rangle$ in TS)
- `Noff[i, j]`  = observed background counts = `solid_angle × bkg[i, 2]`  

Bins where the background count is < 1 are excluded (SWGO below threshold).


In [7]:
n_bins  = len(bkg)
n_mass  = len(mass_used)

Ns   = np.zeros((n_bins, n_dwarfs, n_mass))
Noff = np.zeros((n_bins, n_dwarfs))

for j in range(n_dwarfs):
    # Expected signal: J × gen_ns    [counts per (sigma_v)]
    signal_j  = j_factor[j] * gen_ns          # shape: (n_bins, n_mass)
    # Background in the ROI: bkg counts/sr × solid angle [sr]
    bkg_j     = omega_roi * bkg[:, 2]          # shape: (n_bins,)

    for i in range(n_bins):
        if bkg_j[i] < 1.0:
            # Too few background events — exclude this bin
            Ns[i, j, :] = 0.0
            Noff[i, j]  = 1.0
        else:
            Ns[i, j, :]  = signal_j[i, :]
            Noff[i, j]   = bkg_j[i]

print(f"Ns   shape: {Ns.shape}   (bins × dwarfs × mass)")
print(f"Noff shape: {Noff.shape} (bins × dwarfs)")


Ns   shape: (35, 14, 30)   (bins × dwarfs × mass)
Noff shape: (35, 14) (bins × dwarfs)


## Individual-dwarf upper limits

In [8]:
print("Computing individual-dwarf sigma_v upper limits...")
# Parallelise over dwarfs (each dwarf is fully independent)
sing_sigma_list = Parallel(n_jobs=N_JOBS)(
    delayed(sing_sigma_calc)(Ns[:, j, :], Noff[:, j])
    for j in range(n_dwarfs)
)
# sing_sigma_list[j] has shape (n_mass,)
# We want shape (n_mass, n_dwarfs)
sing_sigma = np.column_stack(sing_sigma_list)
print(f"  Done.  Shape: {sing_sigma.shape}  (n_mass, n_dwarfs)")


Computing individual-dwarf sigma_v upper limits...
  Done.  Shape: (30, 14)  (n_mass, n_dwarfs)


In [9]:
outfile = os.path.join(RESULTS_DIR, f'sing_sigmav_{CHANNEL}.txt')
np.savetxt(outfile, sing_sigma)
print(f"Individual-dwarf limits saved to: {outfile}")
print(f"  Rows  = DM mass points  ({len(mass_used)} values, starting at {INITIAL_MASS} GeV)")
print(f"  Cols  = dwarfs ({n_dwarfs}: {', '.join(order)})")


Individual-dwarf limits saved to: results/sigma/sing_sigmav_b.txt
  Rows  = DM mass points  (30 values, starting at 500 GeV)
  Cols  = dwarfs (14: Carina, Fornax, LeoI, LeoII, Sculptor, Sextans, BootesI, ComaBerenices, Hercules, LeoIV, LeoV, LeoT, Segue1, Segue2)


## Combined upper limit

In [10]:
print("Computing combined sigma_v upper limit...")
combined = sigma_calc(Ns, Noff)
print(f"  Done.  Shape: {combined.shape}  (n_mass,)")


Computing combined sigma_v upper limit...
  Done.  Shape: (30,)  (n_mass,)


In [11]:
outfile = os.path.join(RESULTS_DIR, f'sigmav_{CHANNEL}.txt')
np.savetxt(outfile, combined)
print(f"Combined limit saved to: {outfile}")
print(f"  Length = {len(combined)} DM mass points, starting at {INITIAL_MASS} GeV")


Combined limit saved to: results/sigma/sigmav_b.txt
  Length = 30 DM mass points, starting at 500 GeV
